# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id` values.

We'll list each record set and show the fields and columns (by `@id`) available for exploration.

In [ ]:
print("Available record sets and their fields/columns:\n")
record_sets = []
for rs in dataset.record_sets:
    print(f"RecordSet @id: {rs.id}, Name: {rs.name}")
    print("  Fields:")
    for f in rs.fields:
        print(f"    Field @id: {f.id}, Name: {f.name}, DataType: {f.data_type}")
    print("  Columns:")
    for c in rs.columns:
        print(f"    Column @id: {c.id}, Name: {c.name}, Field: {c.field}")
    print()
    record_sets.append(rs.id)

# For demonstration, assign the first record_set @id
if record_sets:
    first_record_set_id = record_sets[0]
else:
    first_record_set_id = None

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Extract data from each RecordSet by their @id
dataframes = {}

for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for RecordSet @id: {record_set_id}")

# Examine the first record set (or update for target set):
if first_record_set_id:
    print(f"\nColumns in DataFrame for RecordSet {first_record_set_id}:")
    print(dataframes[first_record_set_id].columns.tolist())
    display(dataframes[first_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This can include removing outliers, transforming data, or grouping by a key field (`@id`).

We'll automatically select a numeric field based on the Field's `data_type` and perform basic analysis.

In [ ]:
import numpy as np

if not first_record_set_id:
    print("No record sets available in the metadata to perform EDA.")
else:
    # Find a numeric field from the fields of the RecordSet
    rs_obj = None
    for rs in dataset.record_sets:
        if rs.id == first_record_set_id:
            rs_obj = rs
            break

    numeric_field_id = None
    group_field_id = None

    # Identify numeric and group fields by type and existence in DataFrame columns
    if rs_obj is not None:
        for f in rs_obj.fields:
            # Check for typical numeric datatypes
            if f.data_type in ['Float', 'Double', 'Integer', 'Number', 'schema:Float', 'schema:Double', 'schema:Integer', 'schema:Number']:
                if f.id in dataframes[first_record_set_id].columns:
                    numeric_field_id = f.id
                    break

        # If available, select a non-numeric field for grouping
        for f in rs_obj.fields:
            if f.id != numeric_field_id and f.id in dataframes[first_record_set_id].columns:
                group_field_id = f.id
                break

    df = dataframes[first_record_set_id]
    if numeric_field_id is None:
        print(f"No numeric field found for RecordSet {first_record_set_id}.")
    else:
        print(f"Selected numeric field: {numeric_field_id}")
        # Convert to float if not already
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        threshold = df[numeric_field_id].mean()  # Example threshold: mean

        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()

        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        if group_field_id:
            print(f"\nGrouping by field: {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if first_record_set_id and numeric_field_id:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), bins=40, kde=True)
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.show()
    
    if group_field_id:
        plt.figure(figsize=(10, 5))
        # For large cardinality, take top N categories only
        top_groups = df[group_field_id].value_counts().nlargest(10).index
        plot_df = df[df[group_field_id].isin(top_groups)]
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=plot_df)
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset was loaded and explored using the Croissant metadata and the `mlcroissant` library.
- Record sets, fields, and columns were identified by their `@id` to ensure consistent referencing.
- Basic data cleaning, normalization, and visualization were performed on a numeric field.
- Further analysis can include domain-specific feature engineering, exploratory statistics, and more advanced visualization.
